# 06 — Social charts (Pillow)

Three publication-ready charts, each rendered **inline** (then saved to
`outputs/social/`). Workspace rule: `display()` first, then save — a chart is
never saved without being shown.

1. Worldwide gross — domestic vs international (dumbbell)
2. Which genres travel — international vs domestic index (diverging bars)
3. Biggest *domestic* films, adjusted for inflation (dumbbell)

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

WORKSPACE = PROJECT.parent.parent
sys.path.insert(0, str(WORKSPACE / 'shared'))

import duckdb
from src.ingest import load_config
from chart_templates import lollipop, diverging_bars
from viz import PRESETS
from IPython.display import display

cfg = load_config('config.yaml')
# read-only: viz notebooks only read, so they run alongside an open kernel.
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)
img_w, img_h, _ = PRESETS['twitter_landscape']
out = Path(cfg['paths']['outputs_social']); out.mkdir(parents=True, exist_ok=True)
def money(v):
    return f'${v/1e9:.2f}B' if abs(v) >= 1e9 else f'${v/1e6:.0f}M'

## 1. Where the money comes from: overseas
Top 15 by worldwide gross. Gold = domestic (US/Canada), teal = international.

In [ ]:
ww = con.execute('''SELECT title, release_year, domestic_gross, foreign_gross, worldwide_gross
    FROM films_worldwide ORDER BY worldwide_gross DESC LIMIT 15''').df()
ww['label'] = ww['title'] + '  (' + ww['release_year'].astype(str) + ')'
img2 = lollipop(ww, category_col='label', value_col='foreign_gross', value2_col='domestic_gross',
    label_col='worldwide_gross',  # rows are ranked by worldwide total; label that so it reads as sorted
    value_fmt=money, title='Where the money really comes from: overseas',
    subtitle='Top 15 films by worldwide gross (nominal $), highest first. Teal = international (rest of world), gold = domestic (U.S. & Canada); label = worldwide total.',
    source='Box Office Mojo, Top Lifetime Grosses (Worldwide) — as of Sep 2026',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='International', value2_label='Domestic (US/Canada)',
    img_width=img_w, img_height=img_h)
display(img2)
img2.save(out / '02_worldwide_domestic_vs_international.png')

## 2. Which genres travel?
Each genre's share of international box office relative to domestic (top-200
worldwide films, genres with 10+ films). Right = over-indexes internationally.

In [ ]:
genre = con.execute('''
    WITH fg AS (SELECT w.domestic_gross, w.foreign_gross, g.genre
                FROM films_worldwide w JOIN film_genres_long g
                  ON g.title=w.title AND g.release_year=w.release_year),
         tot AS (SELECT SUM(domestic_gross) d, SUM(foreign_gross) f FROM fg),
         bygenre AS (SELECT genre, COUNT(*) n, SUM(domestic_gross) dom, SUM(foreign_gross) intl
                     FROM fg GROUP BY genre HAVING COUNT(*)>=10)
    SELECT genre AS category,
           ROUND((intl/(SELECT f FROM tot))/NULLIF(dom/(SELECT d FROM tot),0)-1,3) AS value
    FROM bygenre ORDER BY value DESC''').df()
genre['label'] = genre['value'].apply(lambda v: f"{'+' if v>=0 else ''}{v*100:.0f}%")
img3 = diverging_bars(genre, category_col='category', value_col='value', label_col='label',
    title='Every blockbuster genre earns most abroad — but some lean more than others',
    subtitle='Top-200 worldwide films. All earn 62-68% overseas; bars show lean vs. that norm. Right = leans MORE international; left = relatively more domestic.',
    source='Box Office Mojo (Worldwide) + TMDB genres — as of Sep 2026',
    pos_color='#005F73', neg_color='#AE2012', img_width=img_w, img_height=img_h)
display(img3)
img3.save(out / '03_genre_international_vs_domestic_index.png')

## 3. Biggest *domestic* films, adjusted for inflation
A different question: within the U.S. & Canada, adjusted for ticket-price
inflation. Teal = adjusted (2022 $), gold = nominal (release $).

In [ ]:
dom = con.execute('''SELECT title, adjusted_gross, nominal_gross, release_year
    FROM films_adjusted ORDER BY adjusted_gross DESC LIMIT 15''').df()
dom['label'] = dom['title'] + '  (' + dom['release_year'].astype(str) + ')'
img1 = lollipop(dom, category_col='label', value_col='adjusted_gross', value2_col='nominal_gross',
    value_fmt=money, title='The biggest DOMESTIC films of all time, adjusted for inflation',
    subtitle='U.S. & Canada only. Teal = adjusted to 2022 $, gold = nominal (release $) — the gap is a century of ticket-price inflation.',
    source='Box Office Mojo, Top Lifetime Adjusted Grosses (domestic, adj. to 2022) — as of Sep 2026',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Adjusted (2022 $)', value2_label='Nominal (release $)',
    img_width=img_w, img_height=img_h)
display(img1)
img1.save(out / '01_domestic_adjusted_vs_nominal.png')

---
Three charts written to `outputs/social/`. Watermark `@unwelcomedata`, brand
palette, `twitter_landscape` preset.

## Cleanup
Close the DuckDB connection so the lock is released for other tools.

In [ ]:
con.close()
print('connection closed')